# ThreadLearn — Eval: Merged Model
`anha12/threadlearn-qwen2.5-coder-1.5b-merged` (full merged weights, 2B params)

20 real bugs từ production npm packages.

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes huggingface_hub

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
import json, time

secrets = UserSecretsClient()
login(token=secrets.get_secret("HF_TOKEN"))
print("✅ Logged in to Hugging Face")

In [ ]:
MODEL_ID = "anha12/threadlearn-qwen2.5-coder-1.5b-merged"

print(f"📥 Loading {MODEL_ID} ...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.eval()
print("✅ Model loaded")

In [ ]:

REAL_WORLD_CASES = [
    {
        "id": "rw_01",
        "category": "Race Condition",
        "source": "github.com/request/request/issues/2484",
        "code": "fs.exists(filePath, function(exists) {\n  if (exists) {\n    fs.readFile(filePath, 'utf8', callback);\n  }\n});",
        "pass_keywords": [
            "readfile",
            "enoent",
            "atomic",
            "access",
            "toctou",
            "race"
        ]
    },
    {
        "id": "rw_02",
        "category": "Double Callback",
        "source": "github.com/mysqljs/mysql/issues/1260",
        "code": "function query(sql, cb) {\n  connection.connect(function(err) {\n    if (err) cb(err);\n    connection.query(sql, function(err, results) {\n      cb(err, results);\n    });\n  });\n}",
        "pass_keywords": [
            "return cb",
            "return callback",
            "double",
            "called twice",
            "early return"
        ]
    },
    {
        "id": "rw_03",
        "category": "Zalgo",
        "source": "github.com/caolan/async/issues/1066",
        "code": "function getData(key, callback) {\n  if (cache[key]) {\n    callback(null, cache[key]);\n  } else {\n    fetchFromDB(key, function(err, data) {\n      cache[key] = data;\n      callback(err, data);\n    });\n  }\n}",
        "pass_keywords": [
            "nexttick",
            "process.nexttick",
            "setimmediate",
            "zalgo",
            "async",
            "consistent"
        ]
    },
    {
        "id": "rw_04",
        "category": "Event Loop Blocking",
        "source": "github.com/expressjs/express/issues/2471",
        "code": "app.use(function(req, res, next) {\n  const result = [];\n  for (let i = 0; i < req.body.items.length; i++) {\n    result.push(heavyTransform(req.body.items[i]));\n  }\n  res.json(result);\n});",
        "pass_keywords": [
            "worker",
            "setimmediate",
            "async",
            "chunk",
            "promise",
            "block"
        ]
    },
    {
        "id": "rw_05",
        "category": "Context Loss",
        "source": "github.com/davidbanham/express-async-errors/issues/3",
        "code": "class UserController {\n  constructor() { this.db = new Database(); }\n  async getUser(req, res) {\n    const user = await this.db.find(req.params.id);\n    res.json(user);\n  }\n}\nconst ctrl = new UserController();\napp.get('/user/:id', ctrl.getUser);",
        "pass_keywords": [
            "bind",
            "arrow",
            "this",
            ".bind(ctrl)",
            "=> ctrl"
        ]
    },
    {
        "id": "rw_06",
        "category": "Resource Exhaustion",
        "source": "github.com/brianc/node-postgres/issues/1228",
        "code": "pool.connect(function(err, client, done) {\n  client.query('BEGIN', function(err) {\n    if (err) {\n      client.query('ROLLBACK', function(err) { done(); callback(err); });\n    }\n    client.query(sql, function(err, result) {\n      if (err) {\n        client.query('ROLLBACK', function(err) { done(); });\n        callback(err);\n      }\n      client.query('COMMIT', function(err) { done(); callback(null, result); });\n    });\n  });\n});",
        "pass_keywords": [
            "done()",
            "release",
            "return",
            "finally",
            "leak"
        ]
    },
    {
        "id": "rw_07",
        "category": "Stream Leak",
        "source": "github.com/nodejs/node/issues/24941",
        "code": "function serveFile(req, res) {\n  const readStream = fs.createReadStream(req.params.file);\n  const gzip = zlib.createGzip();\n  readStream.pipe(gzip).pipe(res);\n}",
        "pass_keywords": [
            "pipeline",
            "error",
            "destroy",
            "on('error'",
            "cleanup"
        ]
    },
    {
        "id": "rw_08",
        "category": "Race Condition",
        "source": "github.com/jaredhanson/passport/issues/369",
        "code": "app.post('/login', function(req, res, next) {\n  passport.authenticate('local', function(err, user) {\n    if (err) return next(err);\n    req.logIn(user, function(err) {\n      if (err) return next(err);\n      return res.redirect('/');\n    });\n  })(req, res, next);\n});",
        "pass_keywords": [
            "session",
            "atomic",
            "race",
            "concurrent",
            "await"
        ]
    },
    {
        "id": "rw_09",
        "category": "Unhandled Rejection",
        "source": "github.com/tj/co/issues/185",
        "code": "co(function*() {\n  const conn = yield db.connect();\n  const result = yield conn.query(sql);\n  return result;\n}).then(function(result) {\n  res.json(result);\n});",
        "pass_keywords": [
            "catch",
            "try",
            "rejection",
            ".catch(",
            "error"
        ]
    },
    {
        "id": "rw_10",
        "category": "Resource Exhaustion",
        "source": "github.com/nodejs/node/issues/23267",
        "code": "const readStream = fs.createReadStream('huge-file.csv');\nreadStream.on('data', function(chunk) {\n  const parsed = parseCSVChunk(chunk);\n  writableDB.write(parsed);\n});",
        "pass_keywords": [
            "pause",
            "resume",
            "pipe",
            "backpressure",
            "drain",
            "highwatermark"
        ]
    },
    {
        "id": "rw_11",
        "category": "Double Callback",
        "source": "github.com/luin/ioredis/issues/419",
        "code": "function getWithTimeout(key, timeout, cb) {\n  const timer = setTimeout(function() {\n    cb(new Error('timeout'));\n  }, timeout);\n  client.get(key, function(err, data) {\n    cb(err, data);\n  });\n}",
        "pass_keywords": [
            "cleartimeout",
            "clearTimeout",
            "called twice",
            "return cb",
            "once"
        ]
    },
    {
        "id": "rw_12",
        "category": "Sequential Awaits",
        "source": "nodebestpractices",
        "code": "async function getDashboard(userId) {\n  const user = await db.users.findById(userId);\n  const orders = await db.orders.findByUser(userId);\n  const notifications = await db.notifications.findByUser(userId);\n  return { user, orders, notifications };\n}",
        "pass_keywords": [
            "promise.all",
            "parallel",
            "concurrent",
            "Promise.all"
        ]
    },
    {
        "id": "rw_13",
        "category": "Race Condition",
        "source": "github.com/OptimalBits/bull/issues/1016",
        "code": "queue.process(function(job, done) {\n  if (job.data.status === 'pending') {\n    job.data.status = 'processing';\n    processJob(job.data, function(err, result) {\n      done(err, result);\n    });\n  }\n});",
        "pass_keywords": [
            "atomic",
            "race",
            "lock",
            "redis",
            "transaction",
            "update"
        ]
    },
    {
        "id": "rw_14",
        "category": "Callback Hell",
        "source": "github.com/caolan/async/issues/1122",
        "code": "fs.readFile(configPath, function(err, config) {\n  if (err) return callback(err);\n  db.connect(JSON.parse(config), function(err, conn) {\n    if (err) return callback(err);\n    conn.query(sql, function(err, rows) {\n      if (err) return callback(err);\n      rows.forEach(function(row) {\n        transform(row, function(err, result) {\n          results.push(result);\n        });\n      });\n      callback(null, results);\n    });\n  });\n});",
        "pass_keywords": [
            "async/await",
            "async await",
            "promise",
            "flatten",
            "await"
        ]
    },
    {
        "id": "rw_15",
        "category": "Resource Exhaustion",
        "source": "github.com/axios/axios/issues/1038",
        "code": "async function notifyAllUsers(users) {\n  await Promise.all(\n    users.map(user =>\n      axios.post('/notify', { userId: user.id })\n    )\n  );\n}",
        "pass_keywords": [
            "limit",
            "chunk",
            "batch",
            "p-limit",
            "concurren",
            "slice"
        ]
    },
    {
        "id": "rw_16",
        "category": "Event Loop Blocking",
        "source": "Node.js crypto sync API misuse",
        "code": "app.post('/register', function(req, res) {\n  const hash = crypto.pbkdf2Sync(\n    req.body.password,\n    req.body.username,\n    100000, 64, 'sha512'\n  );\n  db.users.create({ hash }, function(err) {\n    res.json({ ok: true });\n  });\n});",
        "pass_keywords": [
            "pbkdf2",
            "async",
            "worker",
            "promise",
            "block"
        ]
    },
    {
        "id": "rw_17",
        "category": "Zalgo",
        "source": "github.com/caolan/async/issues/1122",
        "code": "function processItems(items, callback) {\n  if (items.length === 0) { callback(null, []); return; }\n  const item = items[0];\n  if (computedCache[item]) {\n    processItems(items.slice(1), function(err, rest) {\n      callback(null, [computedCache[item]].concat(rest));\n    });\n  } else {\n    asyncCompute(item, function(err, result) {\n      computedCache[item] = result;\n      processItems(items.slice(1), function(err, rest) {\n        callback(null, [result].concat(rest));\n      });\n    });\n  }\n}",
        "pass_keywords": [
            "nexttick",
            "setimmediate",
            "async",
            "consistent",
            "zalgo"
        ]
    },
    {
        "id": "rw_18",
        "category": "Unhandled Rejection",
        "source": "github.com/expressjs/express/issues/2700",
        "code": "app.get('/user/:id', async function(req, res) {\n  const user = await db.findUser(req.params.id);\n  res.json(user);\n});",
        "pass_keywords": [
            "try",
            "catch",
            "next(err",
            "rejection",
            ".catch"
        ]
    },
    {
        "id": "rw_19",
        "category": "Race Condition",
        "source": "github.com/nodejs/node/issues/6718",
        "code": "function createFileIfNotExists(filePath, content, callback) {\n  fs.access(filePath, fs.constants.F_OK, function(err) {\n    if (err) {\n      fs.writeFile(filePath, content, callback);\n    } else {\n      callback(null);\n    }\n  });\n}",
        "pass_keywords": [
            "wx",
            "exclusive",
            "atomic",
            "flag",
            "race",
            "toctou"
        ]
    },
    {
        "id": "rw_20",
        "category": "Context Loss",
        "source": "Node.js timers docs",
        "code": "class DataPoller {\n  constructor(interval) {\n    this.data = [];\n    this.interval = interval;\n  }\n  start() {\n    setTimeout(function() {\n      this.data.push(Date.now());\n      setTimeout(arguments.callee, this.interval);\n    }, this.interval);\n  }\n}",
        "pass_keywords": [
            "arrow",
            "bind",
            "this",
            "=>",
            ".bind(this)"
        ]
    },
    {
        "id": "rw_21",
        "category": "Sequential Awaits",
        "source": "github.com/expressjs/session/issues/526",
        "code": "app.post('/logout', (req, res) => {\n  req.logout(() => {\n    req.session.save();\n    res.redirect('/');\n  });\n});",
        "pass_keywords": [
            "await",
            "callback",
            "promise",
            "session.save",
            "then"
        ]
    },
    {
        "id": "rw_22",
        "category": "Double Callback",
        "source": "github.com/caolan/async/issues/559",
        "code": "async.series([\n  function(done) {\n    db.query('SELECT 1', function(err) {\n      done(err);\n      done(err);\n    });\n  },\n  function(done) {\n    done();\n  }\n], callback);",
        "pass_keywords": [
            "return done",
            "return callback",
            "once",
            "called twice",
            "guard"
        ]
    },
    {
        "id": "rw_23",
        "category": "Resource Exhaustion",
        "source": "github.com/sequelize/sequelize/issues/10976",
        "code": "const txns = Array(50).fill(null).map(() =>\n  sequelize.transaction(t =>\n    User.create({ name: 'test' }, { transaction: t })\n  )\n);\nawait Promise.all(txns);",
        "pass_keywords": [
            "limit",
            "chunk",
            "batch",
            "pool",
            "concurren",
            "slice",
            "p-limit"
        ]
    },
    {
        "id": "rw_24",
        "category": "Unhandled Rejection",
        "source": "github.com/Automattic/mongoose/issues/8706",
        "code": "const conn = mongoose.createConnection('mongodb://invalid-host:27017/db');\nconn.on('error', function(err) {\n  console.error('connection error:', err);\n});",
        "pass_keywords": [
            "catch",
            ".catch",
            "try",
            "rejection",
            "promise",
            "await"
        ]
    },
    {
        "id": "rw_25",
        "category": "Race Condition",
        "source": "github.com/redis/node-redis/issues/2685",
        "code": "const subscriptions = ['ch1', 'ch2', 'ch3'];\nconst promises = subscriptions.map(ch =>\n  client.sUnsubscribe(ch)\n);\nawait Promise.all(promises);",
        "pass_keywords": [
            "sequential",
            "await",
            "race",
            "disconnect",
            "series",
            "loop"
        ]
    },
    {
        "id": "rw_26",
        "category": "Event Loop Blocking",
        "source": "github.com/knex/knex/issues/5025",
        "code": "await Promise.all([\n  knex.transaction(t => t('users').forUpdate().select()),\n  knex.transaction(t => t('users').forUpdate().select()),\n  knex.transaction(t => t('users').forUpdate().select())\n]);",
        "pass_keywords": [
            "deadlock",
            "sequential",
            "lock",
            "timeout",
            "series",
            "queue"
        ]
    },
    {
        "id": "rw_27",
        "category": "Sequential Awaits",
        "source": "github.com/expressjs/session/issues/340",
        "code": "app.use(function(req, res, next) {\n  req.session.touch();\n  req.session.userId = req.user.id;\n  next();\n});",
        "pass_keywords": [
            "await",
            "callback",
            "promise",
            "race",
            "async",
            "then"
        ]
    },
    {
        "id": "rw_28",
        "category": "Unhandled Rejection",
        "source": "github.com/Automattic/mongoose/issues/5784",
        "code": "User.insertMany(\n  [{ name: 'Alice' }, { name: '' }],\n  function(err, docs) {\n    if (err) return console.log(err);\n    console.log(docs);\n  }\n);",
        "pass_keywords": [
            "catch",
            ".catch",
            "promise",
            "rejection",
            "try",
            "await"
        ]
    },
    {
        "id": "rw_29",
        "category": "Double Callback",
        "source": "github.com/luin/ioredis/issues/1185",
        "code": "async function transfer(from, to, amount) {\n  const pipeline = client.pipeline();\n  pipeline.decrby(from, amount);\n  pipeline.incrby(to, amount);\n  await pipeline.exec();\n  await pipeline.exec();\n}",
        "pass_keywords": [
            "once",
            "return",
            "called twice",
            "remove",
            "exec once",
            "duplicate"
        ]
    },
    {
        "id": "rw_30",
        "category": "Unhandled Rejection",
        "source": "github.com/expressjs/express/issues/2700",
        "code": "app.use(async function(req, res, next) {\n  const data = await fetchData(req.params.id);\n  req.data = data;\n  next();\n});\n\napp.use(async function(req, res, next) {\n  const result = await processData(req.data);\n  res.json(result);\n});",
        "pass_keywords": [
            "try",
            "catch",
            "next(err",
            "rejection",
            "express-async-errors",
            ".catch"
        ]
    }
]
TOTAL = len(REAL_WORLD_CASES)
print(f"✅ Loaded {TOTAL} real-world test cases")

In [ ]:
def score_response(keywords, response):
    if not response:
        return "fail"
    r = response.lower()
    matched = [kw for kw in keywords if kw.lower() in r]
    has_code = any(tok in r for tok in ["function", "const ", "async", "=>", "return", "await"])
    if len(matched) >= 2 and has_code:
        return "pass"
    elif len(matched) >= 1 or has_code:
        return "partial"
    return "fail"

def run_inference(code):
    prompt = f"Convert to concurrent JavaScript:\n\n{code}\n"
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            temperature=1.0,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()

print("✅ Inference functions ready")

In [ ]:
MODEL_LABEL = "anha12/threadlearn-qwen2.5-coder-1.5b-merged"
print("=" * 70)
print(f"ThreadLearn — Real-World Benchmark")
print(f"Model: {MODEL_LABEL}")
print("=" * 70)

results = []
pass_count = partial_count = fail_count = 0

for tc in REAL_WORLD_CASES:
    print(f"\n[{tc['id']}] {tc['category']}")
    t0 = time.time()
    response = run_inference(tc["code"])
    latency = time.time() - t0
    verdict = score_response(tc["pass_keywords"], response)
    if verdict == "pass":    pass_count += 1;    icon = "✅ PASS   "
    elif verdict == "partial": partial_count += 1; icon = "⚠️  PARTIAL"
    else:                    fail_count += 1;    icon = "❌ FAIL   "
    print(f"  {icon} | {latency:.1f}s")
    print(f"  {response[:200].replace(chr(10), ' ')}...")
    results.append({"id": tc["id"], "category": tc["category"],
                    "verdict": verdict, "latency_s": round(latency, 2),
                    "response_preview": response[:400]})

score = pass_count + partial_count * 0.5
print("\n" + "=" * 70)
print(f"RESULTS — {MODEL_LABEL}")
print(f"  Pass:    {pass_count}/{TOTAL}")
print(f"  Partial: {partial_count}/{TOTAL}")
print(f"  Fail:    {fail_count}/{TOTAL}")
print(f"  Score:   {score:.1f}/{TOTAL}  ({pass_count/TOTAL*100:.0f}% full pass)")
print("=" * 70)

with open("/kaggle/working/eval_merged_results.json", "w") as f:
    json.dump({"model": MODEL_LABEL, "pass": pass_count, "partial": partial_count,
               "fail": fail_count, "score": f"{score:.1f}/{TOTAL}", "results": results}, f, indent=2)
print("\n💾 Saved: /kaggle/working/eval_merged_results.json")